# Positioning & Momentum Tracker (BQuant / BQL)

Your own version of Spectra Markets' SFXPM report — see the [explainer](https://www.spectramarkets.com/amfx/sfxpm-explainer/)
— covering precious metals, copper, USD-related FX, rates, energy, and BTC,
running natively in BQuant off `bql`.

**What's actually retrievable on Bloomberg vs. not:**

| Sub-score | Bloomberg? | How |
|---|---|---|
| CFTC LEVEL (net non-comm % OI, percentile since 2000) | Yes | `COT<GO>` / CFTC index tickers, pulled via BQL |
| CFTC 4-Week Change | Yes | same source, differenced |
| RSI (14d) | Yes | price series via BQL, RSI computed locally |
| Deviation from 20d / 100d MA | Yes | price series via BQL |
| Risk Reversal (1m / 6m) | Yes | vol surface — you supply the exact tickers |

DSI and the subjective "house view" line are dropped entirely from this version —
POSITIONING is now just the average of `cftc_level`, `cftc_4wk_chg`, `rr_1m`, `rr_6m`.
Add them back later (as a manual input column) if you want a subjective overlay.

**All tickers below are placeholders** — `<...>` strings, not real Bloomberg
tickers. Fill them in yourself in the `INSTRUMENTS` config cell below, using
what you confirm on `COT<GO>` (CFTC series) and `OVDV<GO>` / `OMON<GO>` (vol
surface).


## 1. Setup

In [ ]:
import bql
import numpy as np
import pandas as pd

bq = bql.Service()

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)


## 2. Instrument universe

Edit this to change what you track. Every ticker/code field is a placeholder —
replace each `<...>` with what you confirm on the terminal:

- `price_ticker` — Bloomberg ticker for spot/futures price (RSI, MA deviation)
- `cftc_code` — CFTC's numeric contract market code. Bloomberg's CFTC index
  tickers are built as `CFTC<code><suffix> Index` (suffix `NCL`/`NCS` =
  non-comm long/short, `OIN` = total open interest) — confirm the exact code
  and ticker construction on `COT<GO>`. Set to `None` for instruments the
  CFTC doesn't cover at all (e.g. FTSE — it trades on ICE Futures Europe, a
  non-US exchange, so there's no US CFTC report for it).
- `vol_ticker_base` — base for constructing vol-surface risk-reversal tickers
  (e.g. something like `<BASE>25R1M Curncy`) — confirm construction and exact
  tenor/delta convention on `OVDV<GO>` or `OMON<GO>`. Set to `None` for any
  instrument that doesn't have a standard vol-surface RR quote you want to use.
- `invert_cftc_sign` — set True where CFTC reports the *inverse* side of the
  pair you're tracking (e.g. JPY futures are long/short yen vs. USD, which is
  inverted relative to how you'd think about USDJPY spot).


In [ ]:
INSTRUMENTS = {
    "XAU": {
        "label": "Gold",
        "asset_class": "metal",
        "price_ticker": "<GOLD_PRICE_TICKER>",
        "cftc_code": "<GOLD_CFTC_CODE>",
        "vol_ticker_base": "<GOLD_VOL_TICKER_BASE>",
    },
    "XAG": {
        "label": "Silver",
        "asset_class": "metal",
        "price_ticker": "<SILVER_PRICE_TICKER>",
        "cftc_code": "<SILVER_CFTC_CODE>",
        "vol_ticker_base": "<SILVER_VOL_TICKER_BASE>",
    },
    "XPT": {
        "label": "Platinum",
        "asset_class": "metal",
        "price_ticker": "<PLATINUM_PRICE_TICKER>",
        "cftc_code": "<PLATINUM_CFTC_CODE>",
        "vol_ticker_base": None,
    },
    "XPD": {
        "label": "Palladium",
        "asset_class": "metal",
        "price_ticker": "<PALLADIUM_PRICE_TICKER>",
        "cftc_code": "<PALLADIUM_CFTC_CODE>",
        "vol_ticker_base": None,
    },
    "DXY": {
        "label": "USD Index",
        "asset_class": "fx",
        "price_ticker": "<DXY_PRICE_TICKER>",
        "cftc_code": "<DXY_CFTC_CODE>",
        "vol_ticker_base": None,
    },
    "EUR": {
        "label": "EUR/USD",
        "asset_class": "fx",
        "price_ticker": "<EURUSD_PRICE_TICKER>",
        "cftc_code": "<EUR_CFTC_CODE>",
        "vol_ticker_base": "<EURUSD_VOL_TICKER_BASE>",
    },
    "JPY": {
        "label": "USD/JPY",
        "asset_class": "fx",
        "price_ticker": "<USDJPY_PRICE_TICKER>",
        "cftc_code": "<JPY_CFTC_CODE>",
        "vol_ticker_base": "<USDJPY_VOL_TICKER_BASE>",
        "invert_cftc_sign": True,
    },
    "OIL": {
        "label": "WTI Crude Oil",
        "asset_class": "energy",
        "price_ticker": "<WTI_PRICE_TICKER>",
        "cftc_code": "<WTI_CFTC_CODE>",     # Legacy report covers WTI (NYMEX) - confirm on COT<GO>
        "vol_ticker_base": "<WTI_VOL_TICKER_BASE>",  # crude has a usable options vol surface; confirm RR convention on OVDV<GO>
    },
    "HG": {
        "label": "Copper",
        "asset_class": "metal",
        "price_ticker": "<COPPER_PRICE_TICKER>",
        "cftc_code": "<COPPER_CFTC_CODE>",  # Copper - Commodity Exchange Inc.; confirm on COT<GO>
        "vol_ticker_base": None,            # no standard FX-style vol surface; leave manual/None unless you build it
    },
    "AUD": {
        "label": "AUD/USD",
        "asset_class": "fx",
        "price_ticker": "<AUDUSD_PRICE_TICKER>",
        "cftc_code": "<AUD_CFTC_CODE>",     # Australian Dollar - CME; confirm on COT<GO>
        "vol_ticker_base": "<AUDUSD_VOL_TICKER_BASE>",
    },
    "GBP": {
        "label": "GBP/USD",
        "asset_class": "fx",
        "price_ticker": "<GBPUSD_PRICE_TICKER>",
        "cftc_code": "<GBP_CFTC_CODE>",     # British Pound - CME; confirm on COT<GO>
        "vol_ticker_base": "<GBPUSD_VOL_TICKER_BASE>",
    },
    "BTC": {
        "label": "Bitcoin",
        "asset_class": "crypto",
        "price_ticker": "<BTC_PRICE_TICKER>",  # e.g. CME BTC futures continuation
        # CME Bitcoin futures are CFTC-reported, but I'm not confident of the exact
        # Bloomberg CFTC index suffix/categorization Bitcoin falls under (it may not
        # follow the standard Legacy NCL/NCS convention used elsewhere in this
        # notebook) -- verify the ticker construction directly on COT<GO> rather
        # than trusting this placeholder blindly.
        "cftc_code": "<BTC_CFTC_CODE>",
        "vol_ticker_base": None,            # no standard FX-style vol surface on Bloomberg; wire up separately (e.g. Deribit) if wanted
    },
    "TU": {
        "label": "2Y Treasury",
        "asset_class": "rates",
        "price_ticker": "<TU_PRICE_TICKER>",   # CME 2-Year Note futures continuation
        "cftc_code": "<TU_CFTC_CODE>",         # confirm on COT<GO> -- Treasuries are also covered by the
                                                # TFF report (Dealer/Asset Manager/Leveraged Funds/Other), which
                                                # may be a more meaningful split than Legacy Non-Commercial for rates
        "vol_ticker_base": None,
    },
    "FV": {
        "label": "5Y Treasury",
        "asset_class": "rates",
        "price_ticker": "<FV_PRICE_TICKER>",   # CME 5-Year Note futures continuation
        "cftc_code": "<FV_CFTC_CODE>",
        "vol_ticker_base": None,
    },
    "TY": {
        "label": "10Y Treasury",
        "asset_class": "rates",
        "price_ticker": "<TY_PRICE_TICKER>",   # CME 10-Year Note futures continuation
        "cftc_code": "<TY_CFTC_CODE>",
        "vol_ticker_base": None,
    },
    # NOTE: there is no standard "1-Year Treasury future" -- CME's note futures
    # ladder starts at 2-Year (TU above). If you meant something else at the
    # very short end (a SOFR future, a T-Bill future), swap this entry for
    # that instead of filling in a ticker here.
    # "ONEY": {
    #     "label": "1Y ???",
    #     "asset_class": "rates",
    #     "price_ticker": "<TICKER>",
    #     "cftc_code": "<CODE>",
    #     "vol_ticker_base": None,
    # },
}

CFTC_HISTORY_START = "2000-01-01"
ZSCORE_CAP = 3.0
DEVIATION_ZSCORE_LOOKBACK = 504  # ~2 years of trading days


## 3. Scoring math

Same -10..+10 conventions as the source report:
- **Percentile scaling** (CFTC LEVEL): today's value ranked against its own
  history since 2000. 0th percentile → -10, 50th → 0, 100th → +10.
- **Capped z-score scaling** (CFTC 4-Week Change, MA deviation, risk reversals):
  z-score vs. trailing history, capped at ±3 std devs, rescaled to ±10 — so a
  genuine "3 std dev extreme" prints as exactly ±10.
- **RSI**: linear map, 0→-10, 50→0, 100→+10.


In [ ]:
def percentile_score(value, history: pd.Series):
    history = history.dropna()
    if value is None or pd.isna(value) or len(history) < 20:
        return None
    percentile = (history <= value).mean() * 100
    return round((percentile - 50) / 50 * 10, 2)


def zscore_score(value, history: pd.Series, cap: float = ZSCORE_CAP):
    history = history.dropna()
    if value is None or pd.isna(value) or len(history) < 20:
        return None
    mean, std = history.mean(), history.std()
    if std == 0 or pd.isna(std):
        return None
    z = max(min((value - mean) / std, cap), -cap)
    return round(z / cap * 10, 2)


def rsi_score(rsi_value):
    if rsi_value is None or pd.isna(rsi_value):
        return None
    return round((rsi_value - 50) / 50 * 10, 2)


def compute_rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    delta = prices.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    rsi[avg_loss == 0] = 100
    rsi[(avg_loss == 0) & (avg_gain == 0)] = 50
    return rsi


def deviation_from_ma(prices: pd.Series, window: int) -> pd.Series:
    ma = prices.rolling(window).mean()
    return (prices - ma) / ma * 100


def safe_mean(scores: list) -> float:
    available = [s for s in scores if s is not None and not pd.isna(s)]
    if not available:
        return None
    return round(sum(available) / len(available), 2)


## 4. CFTC positioning via BQL

Pulls the Bloomberg CFTC index tickers (`CFTC<code>NCL/NCS/OIN Index`) as daily
series (they only update weekly, Fridays, but BQL will just carry the value
forward on the price field) and reconstructs net non-commercial % of open interest,
matching the public CFTC methodology Spectra describes.


In [ ]:
def fetch_cftc_history_bql(cftc_code: str, start_date: str = CFTC_HISTORY_START) -> pd.DataFrame:
    tickers = {
        "ncl": f"CFTC{cftc_code}NCL Index",
        "ncs": f"CFTC{cftc_code}NCS Index",
        "oin": f"CFTC{cftc_code}OIN Index",
    }

    date_range = bq.func.range(start_date, "0d")
    fields = {name: bq.data.px_last(dates=date_range) for name, ticker in tickers.items()}

    frames = {}
    for name, ticker in tickers.items():
        request = bql.Request(ticker, {name: fields[name]})
        response = bq.execute(request)
        df = bql.combined_df(response)[["DATE", name]].set_index("DATE")
        frames[name] = df[name]

    out = pd.concat(frames.values(), axis=1)
    out.columns = list(frames.keys())
    out = out.dropna(how="all").sort_index()

    out["net_noncomm"] = out["ncl"] - out["ncs"]
    out["net_noncomm_pct_oi"] = out["net_noncomm"] / out["oin"] * 100
    return out


def build_positioning_scores(symbol: str, inst: dict) -> dict:
    cftc_code = inst.get("cftc_code")
    if cftc_code is None:
        # Not every instrument has a CFTC report to draw on (e.g. FTSE
        # trades on a non-US exchange) -- just skip cleanly rather than error.
        return {"cftc_level": None, "cftc_4wk_chg": None}

    try:
        hist = fetch_cftc_history_bql(cftc_code)
    except Exception as exc:
        print(f"  [warn] CFTC fetch failed for {symbol}: {exc}")
        return {"cftc_level": None, "cftc_4wk_chg": None}

    series = hist["net_noncomm_pct_oi"].dropna()
    if inst.get("invert_cftc_sign"):
        series = -series

    if series.empty:
        return {"cftc_level": None, "cftc_4wk_chg": None}

    current_value = series.iloc[-1]
    cftc_level = percentile_score(current_value, series)

    # weekly release, so a "4 week" change is 4 observations back in a series
    # of weekly-updated (though daily-indexed) values -- resample to weekly first
    weekly = series.resample("W-FRI").last().dropna()
    change_4wk = weekly.diff(4)
    cftc_4wk_chg = zscore_score(change_4wk.iloc[-1] if len(change_4wk) else None, change_4wk)

    return {"cftc_level": cftc_level, "cftc_4wk_chg": cftc_4wk_chg}


## 5. Momentum via BQL

Straightforward price pull, RSI and MA deviation computed locally in pandas
(kept out of BQL itself so the exact same formulas as the free/public version
of this tracker apply — easy to sanity-check against a chart on the terminal).


In [ ]:
def fetch_price_history_bql(ticker: str, start_date: str = "2015-01-01") -> pd.Series:
    date_range = bq.func.range(start_date, "0d")
    field = bq.data.px_last(dates=date_range)
    request = bql.Request(ticker, {"px_last": field})
    response = bq.execute(request)
    df = bql.combined_df(response)[["DATE", "px_last"]].set_index("DATE").sort_index()
    return df["px_last"].dropna()


def build_momentum_scores(symbol: str, inst: dict) -> dict:
    try:
        prices = fetch_price_history_bql(inst["price_ticker"])
    except Exception as exc:
        print(f"  [warn] Price fetch failed for {symbol}: {exc}")
        return {"rsi": None, "rsi_raw": None, "dev_20d": None, "dev_100d": None}

    rsi_series = compute_rsi(prices, period=14)
    rsi_raw = rsi_series.iloc[-1] if len(rsi_series) else None
    rsi = rsi_score(rsi_raw)

    dev20_series = deviation_from_ma(prices, 20)
    dev_20d = zscore_score(
        dev20_series.iloc[-1] if len(dev20_series) else None,
        dev20_series.tail(DEVIATION_ZSCORE_LOOKBACK),
    )

    dev100_series = deviation_from_ma(prices, 100)
    dev_100d = zscore_score(
        dev100_series.iloc[-1] if len(dev100_series) else None,
        dev100_series.tail(DEVIATION_ZSCORE_LOOKBACK),
    )

    return {"rsi": rsi, "rsi_raw": rsi_raw, "dev_20d": dev_20d, "dev_100d": dev_100d}


## 6. Risk reversals via BQL (optional — vol surface)

Pulls the 25-delta risk reversal level for instruments that have a
`vol_ticker_base` configured, then z-scores it against its own trailing
history — same treatment Spectra describes ("we look at the z-score of the
risk-reversal... 1-month and 6-month lookback").

Where `vol_ticker_base` is `None` (platinum, palladium, DXY — these don't
trade on a standard FX-style vol surface), this returns `None` and you fill
those cells manually.


In [ ]:
def fetch_risk_reversal_scores(symbol: str, inst: dict) -> dict:
    base = inst.get("vol_ticker_base")
    if base is None:
        return {"rr_1m": None, "rr_6m": None}

    out = {}
    for tenor, key in [("1M", "rr_1m"), ("6M", "rr_6m")]:
        ticker = f"{base}25R{tenor} Curncy"
        try:
            series = fetch_price_history_bql(ticker, start_date="2018-01-01")
            out[key] = zscore_score(series.iloc[-1] if len(series) else None, series)
        except Exception as exc:
            print(f"  [warn] RR fetch failed for {symbol} {tenor} ({ticker}): {exc}")
            out[key] = None
    return out


## 7. Build the report

In [ ]:
rows = []
for symbol, inst in INSTRUMENTS.items():
    print(f"Fetching {symbol} ({inst['label']})...")
    row = {"symbol": symbol, "label": inst["label"], "asset_class": inst["asset_class"]}

    row.update(build_positioning_scores(symbol, inst))
    row.update(build_momentum_scores(symbol, inst))
    row.update(fetch_risk_reversal_scores(symbol, inst))

    row["POSITIONING"] = safe_mean([
        row["cftc_level"], row["cftc_4wk_chg"], row["rr_1m"], row["rr_6m"],
    ])
    row["MOMENTUM"] = safe_mean([row["rsi"], row["dev_20d"], row["dev_100d"]])

    rows.append(row)

report = pd.DataFrame(rows).set_index("symbol")
col_order = [
    "label", "asset_class",
    "POSITIONING", "cftc_level", "cftc_4wk_chg", "rr_1m", "rr_6m",
    "MOMENTUM", "rsi", "rsi_raw", "dev_20d", "dev_100d",
]
report = report[col_order]
report.round(2)


## 8. Flag extremes

In [ ]:
def flag_extremes(df: pd.DataFrame, threshold: float = 5.0) -> None:
    print(f"--- Extremes (|score| >= {threshold}) ---")
    for symbol, row in df.iterrows():
        for col in ("POSITIONING", "MOMENTUM"):
            val = row[col]
            if val is not None and not pd.isna(val) and abs(val) >= threshold:
                direction = "extreme long/bullish" if val > 0 else "extreme short/bearish"
                print(f"  {symbol} {col}: {val:+.1f}  ({direction})")

flag_extremes(report)


## 9. RSI heatmap

A quick visual scan across the whole tracked universe — sorted highest RSI
(overbought / blue) to lowest (oversold / red), same style as the classic
"global RSI heatmap" boards you'll see floating around trading desks.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors


def plot_rsi_heatmap(df: pd.DataFrame, ncols: int = 5, cell_size=(1.9, 1.3)):
    data = df[["label", "rsi_raw"]].dropna(subset=["rsi_raw"]).sort_values("rsi_raw", ascending=False)
    n = len(data)
    if n == 0:
        print("No RSI data available to plot.")
        return

    nrows = -(-n // ncols)  # ceil division
    fig, ax = plt.subplots(figsize=(cell_size[0] * ncols, cell_size[1] * nrows))

    cmap = plt.get_cmap("RdYlBu")  # low RSI -> red, mid -> pale/yellow, high RSI -> blue
    norm = mcolors.Normalize(vmin=0, vmax=100)

    for i, (symbol, row) in enumerate(data.iterrows()):
        r, c = divmod(i, ncols)
        y = nrows - r - 1
        color = cmap(norm(row["rsi_raw"]))

        ax.add_patch(patches.Rectangle((c, y), 1, 1, facecolor=color, edgecolor="white", linewidth=2))

        # pick readable text color against the cell's background
        luminance = 0.299 * color[0] + 0.587 * color[1] + 0.114 * color[2]
        text_color = "white" if luminance < 0.55 else "black"

        label = row["label"] if pd.notna(row["label"]) else symbol
        ax.text(c + 0.5, y + 0.62, label, ha="center", va="center",
                fontsize=10, fontweight="bold", color=text_color)
        ax.text(c + 0.5, y + 0.30, f'{row["rsi_raw"]:.0f}', ha="center", va="center",
                fontsize=11, color=text_color)

    ax.set_xlim(0, ncols)
    ax.set_ylim(0, nrows)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title("Tracked-Instrument RSI Heatmap\nCurrent 14-day RSI", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


plot_rsi_heatmap(report, ncols=5)


## 10. Save

Writes a dated CSV so you can build a time series of your own scores week to
week (useful for eyeballing regime shifts, same as the historical USD/EUR/AUD
charts in the Spectra piece).


In [ ]:
from datetime import date

out_path = f"sfxpm_report_{date.today().isoformat()}.csv"
report.to_csv(out_path)
print(f"Saved to {out_path}")
